# Hyper-parameter Experiments

Experiment with hyper-parameters to improve the "Price is Right" models.

**Task:** Try different settings for Random Forest (and optionally other models), compare validation error, and evaluate the best config on the test set.

In [ ]:
# imports
import sys
from pathlib import Path

root_dir = Path().resolve().parent.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import mean_absolute_error
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
# Authenticate with Hugging Face
load_dotenv(override=True)
if os.environ.get("HF_TOKEN"):
    login(os.environ["HF_TOKEN"], add_to_git_credential=True)

In [ ]:
# setup
LITE_MODE = True 

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}")

In [ ]:
# Helper — validation MAE
def validation_mae(pricer_fn, items, size=500):
    """Mean absolute error of pricer_fn on the first `size` items."""
    subset = items[:size]
    preds = [pricer_fn(item) for item in subset]
    truths = [item.price for item in subset]
    return mean_absolute_error(truths, preds)

In [ ]:
# Random Forest — hyper-parameter sweep
np.random.seed(42)

train_docs = [item.summary for item in train]
train_prices = np.array([item.price for item in train])
val_docs = [item.summary for item in val]
val_prices = np.array([item.price for item in val])

# defaults: max_features=2000, n_estimators=100, no max_depth
param_grid = {
    "max_features": [2000, 5000],
    "n_estimators": [100, 200, 300],
    "max_depth": [15, 25, None],
}

results = []

In [ ]:
from itertools import product

for max_feat, n_est, max_d in product(
    param_grid["max_features"],
    param_grid["n_estimators"],
    param_grid["max_depth"],
):
    vec = CountVectorizer(max_features=max_feat, stop_words="english")
    X_train = vec.fit_transform(train_docs)
    X_val = vec.transform(val_docs)

    model = RandomForestRegressor(
        n_estimators=n_est,
        max_depth=max_d,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, train_prices)
    val_pred = model.predict(X_val)
    mae = mean_absolute_error(val_prices, val_pred)
    results.append({
        "max_features": max_feat,
        "n_estimators": n_est,
        "max_depth": str(max_d),
        "val_mae": round(mae, 2),
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_mae")
results_df

In [ ]:
# Best config — train on train+val and evaluate on test
best = results_df.iloc[0]
max_feat = int(best["max_features"])
n_est = int(best["n_estimators"])
max_d = None if best["max_depth"] == "None" else int(best["max_depth"])
print(f"Best: max_features={max_feat}, n_estimators={n_est}, max_depth={max_d}")

In [ ]:
vectorizer_final = CountVectorizer(max_features=max_feat, stop_words="english")
all_train = train + val
all_docs = [item.summary for item in all_train]
all_prices = np.array([item.price for item in all_train])

X_all = vectorizer_final.fit_transform(all_docs)
rf_final = RandomForestRegressor(n_estimators=n_est, max_depth=max_d, random_state=42, n_jobs=-1)
rf_final.fit(X_all, all_prices)

def best_rf_pricer(item):
    X = vectorizer_final.transform([item.summary])
    return float(rf_final.predict(X)[0])

print("Running full evaluation on test set...")
evaluate(best_rf_pricer, test)

## 5. Summary

Compare the test error above with Week 6 default Random Forest (~$72.28). Note your best validation MAE and whether the tuned model improved on the test set.